# Hillsboro GIS Data Acquisition

## Purpose

This notebook acquires immutable raw snapshots of selected City of Hillsboro GIS datasets for the Web Hosted Portfolio project.

## Raw Data Policy

No attribute or geometry transformations are performed during acquisition.

## Pagination

ArcGIS REST services have dataset-specific transfer limits. The acquisition function retrieves records in batches until the complete layer has been downloaded.

## Versioning

New acquisitions receive a new timestamped snapshot directory rather than overwriting previous snapshots.

## Storage

Large raw JSON files are stored outside GitHub in the project's permanent raw-data storage. GitHub contains project code, documentation, and metadata/manifests.

In [2]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import requests

# --------------------------------------------------
# HIL Raw Data Acquisition
# --------------------------------------------------

SNAPSHOT_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

RAW_ROOT = Path(".")

print(f"Snapshot date: {SNAPSHOT_DATE}")
print(f"Raw data root: {RAW_ROOT}")

Snapshot date: 2026-09-08
Raw data root: .


In [3]:
DATASETS = {
    "HIL-001": {
        "name": "semiconductor_businesses",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "EcDev_SemiconductorBusinesses/FeatureServer/0"
        ),
        "source_organization": "City of Hillsboro"
    },

    "HIL-002": {
        "name": "project_boundaries",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "ContructionProjects/FeatureServer/0"
        ),
    },

    "HIL-003": {
        "name": "zoning",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "LandUseGallery_Prod/FeatureServer/23"
        ),
    },

    "HIL-004": {
        "name": "comprehensive_plan",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "LandUseGallery_Prod/FeatureServer/21"
        ),
    },

    "HIL-005": {
        "name": "buildings",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "Planning_BaseData/MapServer/91"
        ),
    },

    "HIL-006": {
        "name": "metro_buildings",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "Planning_BaseData/MapServer/92"
        ),
    },

    "HIL-007": {
        "name": "pavement_projects",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "ContructionProjects/FeatureServer/1"
        ),
    },

    "HIL-008": {
        "name": "city_limits",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "Hillsboro_City_Limits/FeatureServer/0"
        ),
    },

    "HIL-009": {
        "name": "roadway",
        "source_organization": "City of Hillsboro",
        "url": (
            "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
            "Planning_BaseData/MapServer/80"
        ),
    },
}

print(f"Datasets configured: {len(DATASETS)}")

Datasets configured: 9


In [4]:
def acquire_dataset(dataset_id, config):
    """
    Download a complete ArcGIS REST layer as raw JSON.

    - Retrieves records in batches.
    - Preserves source attributes and geometry.
    - Creates a timestamped snapshot directory.
    - Creates a manifest with metadata and SHA-256 checksum.
    """

    layer_url = config["url"].rstrip("/")
    query_url = f"{layer_url}/query"

    acquired_at = datetime.now(timezone.utc).isoformat()

    # --------------------------------------------------
    # Layer metadata
    # --------------------------------------------------

    layer_response = requests.get(
        layer_url,
        params={"f": "json"},
        timeout=60
    )

    layer_response.raise_for_status()

    layer_info = layer_response.json()

    if "error" in layer_info:
        raise RuntimeError(
            f"Layer metadata error for {dataset_id}: "
            f"{layer_info['error']}"
        )

    max_record_count = layer_info.get(
        "maxRecordCount",
        1000
    )

    print(f"Downloading {dataset_id}...")
    print(f"Query URL: {query_url}")
    print(f"Batch size: {max_record_count}")

    # --------------------------------------------------
    # Retrieve all records
    # --------------------------------------------------

    all_features = []
    offset = 0
    first_batch = None

    while True:

        params = {
            "where": "1=1",
            "outFields": "*",
            "returnGeometry": "true",
            "resultOffset": offset,
            "resultRecordCount": max_record_count,
            "f": "json"
        }

        response = requests.get(
            query_url,
            params=params,
            timeout=120
        )

        response.raise_for_status()

        batch = response.json()

        if first_batch is None:
            first_batch = batch

        if "error" in batch:
            raise RuntimeError(
                f"ArcGIS error for {dataset_id}: "
                f"{batch['error']}"
            )

        features = batch.get("features", [])

        if not features:
            break

        all_features.extend(features)

        print(
            f"  Retrieved {len(all_features):,} records..."
        )

        if not batch.get(
            "exceededTransferLimit",
            False
        ):
            break

        offset += len(features)

    # --------------------------------------------------
    # Validate acquisition
    # --------------------------------------------------

    record_count = len(all_features)

    if record_count == 0:
        raise RuntimeError(
            f"{dataset_id} returned zero records. "
            "No snapshot was saved."
        )

    # --------------------------------------------------
    # Reconstruct complete response
    # --------------------------------------------------

    data = {
        key: value
        for key, value in first_batch.items()
        if key != "features"
    }

    data["features"] = all_features

    # --------------------------------------------------
    # Create snapshot directories
    # --------------------------------------------------

    snapshot_dir = (
        RAW_ROOT /
        SNAPSHOT_DATE
    )

    datasets_dir = (
        snapshot_dir /
        "datasets" /
        "raw"
    )

    manifests_dir = (
        snapshot_dir /
        "manifests"
    )

    datasets_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    manifests_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------
    # Save raw JSON
    # --------------------------------------------------

    data_file = (
        datasets_dir /
        f"{dataset_id}.json"
    )

    with open(
        data_file,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(data, f)

    # --------------------------------------------------
    # SHA-256 checksum
    # --------------------------------------------------

    sha256 = hashlib.sha256(
        data_file.read_bytes()
    ).hexdigest()

    # --------------------------------------------------
    # Manifest
    # --------------------------------------------------

    manifest = {
        "dataset_id": dataset_id,
        "dataset_name": config["name"],
        "source_organization": (
            config["source_organization"]
        ),
        "source_url": layer_url,
        "query_url": query_url,
        "acquired_at_utc": acquired_at,
        "snapshot_date": SNAPSHOT_DATE,
        "record_count": record_count,
        "file": data_file.name,
        "file_size_bytes": data_file.stat().st_size,
        "sha256": sha256,
        "notes": (
            "Raw acquisition snapshot. "
            "No attribute or geometry "
            "transformations performed."
        )
    }

    manifest_file = (
        manifests_dir /
        f"{dataset_id}_manifest.json"
    )

    with open(
        manifest_file,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            manifest,
            f,
            indent=2
        )
        f.write("\n")

    print("\n✓ Complete acquisition")
    print(f"✓ Records: {record_count:,}")
    print(f"✓ Saved: {data_file}")
    print(f"✓ Manifest: {manifest_file}")

    return manifest

In [5]:
# Used when downloading a single dataset
manifest = acquire_dataset(
    "HIL-001",
    DATASETS["HIL-001"]
)

Query URL: https://gis.hillsboro-oregon.gov/public/rest/services/public/EcDev_SemiconductorBusinesses/FeatureServer/0/query
Batch size: 2000
  Retrieved 59 records...

✓ Complete acquisition
✓ Records: 59
✓ Saved: 2026-09-08\datasets\raw\HIL-001.json
✓ Manifest: 2026-09-08\manifests\HIL-001_manifest.json


In [6]:
datasets_to_acquire = [
     "HIL-001",
     "HIL-002",
     "HIL-003",
     "HIL-004",
     "HIL-005",
     "HIL-006",
     "HIL-007",
     "HIL-008",
     "HIL-009",
 ]

for dataset_id in datasets_to_acquire:
     print("\n" + "=" * 70)
     acquire_dataset(
         dataset_id,
         DATASETS[dataset_id]
     )


Query URL: https://gis.hillsboro-oregon.gov/public/rest/services/public/EcDev_SemiconductorBusinesses/FeatureServer/0/query
Batch size: 2000
  Retrieved 59 records...

✓ Complete acquisition
✓ Records: 59
✓ Saved: 2026-09-08\datasets\raw\HIL-001.json
✓ Manifest: 2026-09-08\manifests\HIL-001_manifest.json

Query URL: https://gis.hillsboro-oregon.gov/public/rest/services/public/ContructionProjects/FeatureServer/0/query
Batch size: 2000
  Retrieved 145 records...

✓ Complete acquisition
✓ Records: 145
✓ Saved: 2026-09-08\datasets\raw\HIL-002.json
✓ Manifest: 2026-09-08\manifests\HIL-002_manifest.json

Query URL: https://gis.hillsboro-oregon.gov/public/rest/services/public/LandUseGallery_Prod/FeatureServer/23/query
Batch size: 2000
  Retrieved 353 records...

✓ Complete acquisition
✓ Records: 353
✓ Saved: 2026-09-08\datasets\raw\HIL-003.json
✓ Manifest: 2026-09-08\manifests\HIL-003_manifest.json

Query URL: https://gis.hillsboro-oregon.gov/public/rest/services/public/LandUseGallery_Prod/F

In [7]:
# ============================================================
# HIL RAW DATA VALIDATION
# ============================================================

from pathlib import Path
import json
import hashlib

DATA_ROOT = Path(".")
SNAPSHOT_DIR = DATA_ROOT / SNAPSHOT_DATE
DATASETS_DIR = SNAPSHOT_DIR / "datasets" / "raw"
MANIFESTS_DIR = SNAPSHOT_DIR / "manifests"

print("HIL RAW DATA VALIDATION")
print("=" * 70)
print(f"Snapshot: {SNAPSHOT_DATE}")
print(f"Datasets: {DATASETS_DIR}")
print(f"Manifests: {MANIFESTS_DIR}")
print()

dataset_ids = [
    "HIL-001",
    "HIL-002",
    "HIL-003",
    "HIL-004",
    "HIL-005",
    "HIL-006",
    "HIL-007",
    "HIL-008",
    "HIL-009",
]

for dataset_id in dataset_ids:

    data_file = DATASETS_DIR / f"{dataset_id}.json"
    manifest_file = MANIFESTS_DIR / f"{dataset_id}_manifest.json"

    print(f"{dataset_id}")

    # --------------------------------------------------------
    # JSON
    # --------------------------------------------------------

    if not data_file.exists():
        print("  ✗ JSON: missing")
        continue

    size = data_file.stat().st_size

    with open(data_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    records = len(data.get("features", []))

    # --------------------------------------------------------
    # SHA-256
    # --------------------------------------------------------

    sha256 = hashlib.sha256(
        data_file.read_bytes()
    ).hexdigest()

    print(f"  ✓ JSON: {size:,} bytes")
    print(f"  ✓ Records: {records:,}")
    print(f"  ✓ SHA-256: {sha256[:16]}...")

    # --------------------------------------------------------
    # Manifest
    # --------------------------------------------------------

    if manifest_file.exists():
        print("  ✓ Manifest: present")
    else:
        print("  ✗ Manifest: missing")

    print()

print("=" * 70)
print("Validation complete.")

HIL RAW DATA VALIDATION
Snapshot: 2026-09-08
Datasets: 2026-09-08\datasets\raw
Manifests: 2026-09-08\manifests

HIL-001
  ✓ JSON: 52,556 bytes
  ✓ Records: 59
  ✓ SHA-256: a97731f7930c7c74...
  ✓ Manifest: present

HIL-002
  ✓ JSON: 693,313 bytes
  ✓ Records: 145
  ✓ SHA-256: d9d1c89b722d7c33...
  ✓ Manifest: present

HIL-003
  ✓ JSON: 1,433,918 bytes
  ✓ Records: 353
  ✓ SHA-256: 5135b28e0db157c0...
  ✓ Manifest: present

HIL-004
  ✓ JSON: 2,383,185 bytes
  ✓ Records: 297
  ✓ SHA-256: a3f3d0a661df2959...
  ✓ Manifest: present

HIL-005
  ✓ JSON: 53,675,278 bytes
  ✓ Records: 43,762
  ✓ SHA-256: 3f91f55e48c6a9d5...
  ✓ Manifest: present

HIL-006
  ✓ JSON: 220,649,333 bytes
  ✓ Records: 138,063
  ✓ SHA-256: 9246cf61e4cf88db...
  ✓ Manifest: present

HIL-007
  ✓ JSON: 105,915 bytes
  ✓ Records: 66
  ✓ SHA-256: 4f88834a6031aafd...
  ✓ Manifest: present

HIL-008
  ✓ JSON: 199,849 bytes
  ✓ Records: 27
  ✓ SHA-256: dc4ccfedd5612901...
  ✓ Manifest: present

HIL-009
  ✓ JSON: 5,644,707 bytes
